# processed_combined — 국민여행조사 2023-2025 통합 전처리

**베이스(뼈대): 진경 노트북(`combined_2023_2024_2025_전처리.ipynb`)**
결측치/타입/이상치(IQR×3)/손상레코드(ID 제외 중복검사 포함)/거주지 복원(RESIDENCE_SIDO)/공식통계 대조를 그대로 채택.
원본 82개 컬럼 값은 하나도 바꾸지 않고, 행도 삭제하지 않는다 (진경 원칙 그대로 유지).

**표면: 승희 노트북(`eda_seunghee.ipynb`)**
방문 시/도 파생, 인구통계 라벨링, 지역×월 교차변수, 컬럼명 영문화 규칙을 채택.
(`peak_season` 계열 파생변수는 트랙1·트랙2 방법론 문서 어디에도 요구사항이 없어 이번 통합에서는 제외하기로 팀에서 결정함)

## 채택 근거 요약 (대화 검증 결과)

| 항목 | 채택 | 근거 |
|---|---|---|
| dtype 변환 | 진경 | 컬럼별 소수점 실제 존재 여부를 전수 감사 후 결정(승희는 추측 기반) |
| 이상치(D_TRA1_COST/NUM) | 진경 (IQR×3, 값 불변 플래그만) | IQR×1.5(승희)로 잡히는 경계구간의 77%가 2~4인 정상 가족여행(1인당 지출 24만원 수준)으로 확인 — 과검출 |
| 완전중복행 검사 | 진경 (ID 제외하고 검사) | ID 포함 검사(승희)는 ID가 응답자 고유값이라 애초에 0건만 나오는 무의미한 검사. 결론(삭제 안 함)은 동일 |
| 손상레코드(회차응답수 vs 지역합) | 동일 수준, 우열 없음 | 둘 다 다지역 여행으로 원인 규명, 삭제 0건 |
| 거주지 파생(RESIDENCE_SIDO) | 진경, 필수 | 광역시 방문 왜곡 문제(문제1) 검증의 전제. SPOT 코드와 교차검증 일치율 100% |
| SPOT→시/군/구 디코딩 | 신규 작업 | 둘 다 안 만들었음. 트랙2 방법론이 요구하는 항목이라 코드북 매핑표로 이번에 추가 |
| 방문 시/도, 인구통계 라벨, 지역×월 교차변수 | 승희 | 분석·챗봇에 바로 쓰는 파생변수 |
| peak_season | **제외** | 트랙1·트랙2 방법론 문서 어디에도 요구사항 없음 (팀 결정) |
| 컬럼명 | 승희(영문 snake_case) | 팀 네이밍 규칙, 원본 컬럼까지 전부 영문화 |

## 이번 통합에서 추가한 것
- `trip1_dest_sigungu_cd`, `trip1_dest_sigungu_nm` — SPOT코드(5자리) → 코드북 "시도 및 시군구 코드" 표를 join한 시/군/구 단위 방문지 (트랙2 방법론 2-2 ②가 요구하나 두 원본 노트북 모두 미구현이었던 항목)
- `residence_region_type` — 거주지가 광역시인지 도인지 표시. 국민여행조사 유저가이드 141쪽 공식 정의(광역시는 구간 이동이 여행으로 안 잡히고, 도는 시/군간 이동이 여행으로 잡힘)에 따라 방문건수 척도가 서로 달라, 트랙1 4분면 분석 시 광역시/도를 층화해서 비교해야 하기 때문에 추가

## 처리 순서
0 준비 → 1 결측치 → 2 타입변환 → 3 이상치 → 4 손상레코드 → 5 파생변수(거주지·SPOT디코딩·방문시도·인구라벨·교차변수) → 6 공식통계 대조 → 7 영문 컬럼명 변경 → 8 저장(`processed_combined.csv`)


---
## 0. 준비

In [1]:
import json

import numpy as np
import pandas as pd

SRC = 'combined_2023_2024_2025.csv'
DST = 'processed_combined.csv'
SIGUNGU_MAP_PATH = 'sigungu_map.json'

REGIONS = ['서울','부산','대구','인천','광주','대전','울산','세종','경기','강원',
           '충북','충남','전북','전남','경북','경남','제주']

CNT  = [f'국내_A_여행횟수_관광전체_{r}' for r in REGIONS]   # 지역별 방문 횟수 17개
CST  = [f'국내_A_여행지출_관광전체_{r}' for r in REGIONS]   # 지역별 지출 17개
CASE = [f'D_TRA{n}_CASE' for n in range(1, 7)]              # 회차별 여행 목적

INT_TRA = [f'D_TRA{n}_{s}' for n in range(1, 7)
           for s in ['SYEAR','SMONTH','1_SPOT','CASE','COST','NUM','ONE_COST']]

TOUR_CASE = [1, 2, 4]        # 관광여행으로 인정하는 CASE 값 (1=관광/휴양, 2=친지방문+관광, 4=출장+관광)

METRO = ['서울','부산','대구','인천','광주','대전','울산','세종']  # 8개 광역시/특별자치시

# ID 앞 2자리 -> 시도 매핑 (SPOT 코드 체계와 교차대조로 100% 일치 확인된 매핑, 진경 노트북 검증 결과)
SIDO = {'11':'서울','21':'부산','22':'대구','23':'인천','24':'광주','25':'대전','26':'울산',
        '29':'세종','31':'경기','32':'강원','33':'충북','34':'충남','35':'전북','36':'전남',
        '37':'경북','38':'경남','39':'제주'}

df = pd.read_csv(SRC, low_memory=False)
print(f'원본 : {df.shape[0]:,}행 x {df.shape[1]}열')
print('연도별 :', df['연도'].value_counts().sort_index().to_dict())
print('ID 중복 :', df['ID'].duplicated().sum(), '건')

step_log = [('원본', df.shape[0], df.shape[1])]

원본 : 156,050행 x 82열
연도별 : {2023: 52111, 2024: 51754, 2025: 52185}
ID 중복 : 0 건


---
## 1. 결측치

`D_TRA1~6`(회차별 반복필드)이 비어 있는 사람은 "여행을 하지 않음"에 따른 구조적 결측이다.
0으로 채우면 평균이 왜곡되므로 원본 NaN을 그대로 유지한다.

In [2]:
print('[핵심 키] 결측')
for c in ['ID','WT_DOM','연도','BSEX','BAGE','BINC1']:
    n = int(df[c].isna().sum())
    assert n == 0, f'{c} 결측 {n}건 발견'
    print(f'  {c:8s} 결측 0건')

print()
print('[지역 사전집계 34컬럼] 결측', int(df[CNT].isna().sum().sum() + df[CST].isna().sum().sum()), '건')

no_tra1 = df['D_TRA1_SYEAR'].isna()
print()
print(f'D_TRA1 결측 인원 : {no_tra1.sum():,}명')
print(f'  그중 지역 사전집계도 전부 0 : {(no_tra1 & (df[CNT].sum(axis=1) == 0)).sum():,}명 -> 구조적 결측(비여행자)')

# 참고: 0으로 채웠을 때 평균이 왜곡되는지 확인 (실제로 적용하지 않음)
real_mean = df.loc[df['D_TRA1_COST'].notna(), 'D_TRA1_COST'].mean()
zero_mean = df['D_TRA1_COST'].fillna(0).mean()
print(f'[참고] D_TRA1_COST 평균 — 결측 제외 {real_mean:,.0f}원 vs 0으로 채움 {zero_mean:,.0f}원 '
      f'({zero_mean/real_mean-1:+.1%}) -> 채우지 않는다')

step_log.append(('결측치 처리', df.shape[0], df.shape[1]))
print()
print('결측치 처리: 없음 (원본 NaN 유지)')

[핵심 키] 결측
  ID       결측 0건
  WT_DOM   결측 0건
  연도       결측 0건
  BSEX     결측 0건
  BAGE     결측 0건
  BINC1    결측 0건

[지역 사전집계 34컬럼] 결측 0 건

D_TRA1 결측 인원 : 76,320명
  그중 지역 사전집계도 전부 0 : 75,985명 -> 구조적 결측(비여행자)
[참고] D_TRA1_COST 평균 — 결측 제외 301,721원 vs 0으로 채움 154,157원 (-48.9%) -> 채우지 않는다

결측치 처리: 없음 (원본 NaN 유지)


---
## 2. 타입 변환

컬럼별로 소수점이 실제로 존재하는지 전수 감사한 뒤 결정한다 (승희처럼 "결측 때문일 것"이라 추측하지 않음).

In [3]:
rows = []
for c in df.columns:
    if not pd.api.types.is_numeric_dtype(df[c]):
        continue
    s = df[c].dropna().astype('float64')
    rows.append({'컬럼': c, '소수점셀': int((s % 1 != 0).sum())})
audit = pd.DataFrame(rows)
need_float = audit[audit['소수점셀'] > 0]
print(f'소수점이 실제로 존재해 float를 유지해야 하는 컬럼 : {len(need_float)}개')
print(need_float.to_string(index=False))
print(f'나머지 {len(audit) - len(need_float)}개는 전부 정수값 -> 정수형 변환 가능')

소수점이 실제로 존재해 float를 유지해야 하는 컬럼 : 18개
               컬럼   소수점셀
           WT_DOM 156043
국내_A_여행지출_관광전체_서울     45
국내_A_여행지출_관광전체_부산     48
국내_A_여행지출_관광전체_대구     30
국내_A_여행지출_관광전체_인천     15
국내_A_여행지출_관광전체_광주     58
국내_A_여행지출_관광전체_대전     48
국내_A_여행지출_관광전체_울산     41
국내_A_여행지출_관광전체_세종     16
국내_A_여행지출_관광전체_경기     97
국내_A_여행지출_관광전체_강원    112
국내_A_여행지출_관광전체_충북     50
국내_A_여행지출_관광전체_충남     88
국내_A_여행지출_관광전체_전북    102
국내_A_여행지출_관광전체_전남    173
국내_A_여행지출_관광전체_경북    125
국내_A_여행지출_관광전체_경남    140
국내_A_여행지출_관광전체_제주      7
나머지 63개는 전부 정수값 -> 정수형 변환 가능


In [4]:
# 회차 42개는 Int64(대문자, nullable)로 변환 — 구조적 결측을 표현해야 하므로 소문자 int64는 불가
for c in INT_TRA:
    df[c] = df[c].round().astype('Int64')

# 지역 사전집계 횟수 17개 (결측 없음)
for c in CNT:
    df[c] = df[c].astype('int64')

df['연도'] = df['연도'].astype('int64')

# 지출 17개: 소수점이 있는 셀은 1.6%뿐이고 다지역 여행 분할배분에 의한 잔차. 반올림 오차가 무시 가능한 수준임을
# 확인한 뒤 정수로 통일 (WT_DOM만 float 유지 — 가중치는 소수점이 본질)
spend_before = df[CST].values.sum()
for c in CST:
    df[c] = df[c].round().astype('int64')
spend_after = df[CST].values.sum()

err_pct = (spend_after - spend_before) / spend_before
print(f'지출 컬럼 반올림 오차: {spend_after - spend_before:,.0f}원 ({err_pct:.9%})')
assert abs(err_pct) < 1e-6, '지출 반올림 오차가 예상보다 큼 — 확인 필요'

step_log.append(('타입 변환', df.shape[0], df.shape[1]))
print()
print('dtype 분포 :', df.dtypes.value_counts().to_dict())

지출 컬럼 반올림 오차: 9원 (0.000000091%)

dtype 분포 : {Int64Dtype(): 42, dtype('int64'): 38, <StringDtype(storage='python', na_value=nan)>: 1, dtype('float64'): 1}


---
## 3. 이상치

연도별로, 여행을 한 사람만 대상으로 **IQR × 3 (Outer fence)** 상한을 넘는 값을 이상치로 플래그한다 (값은 바꾸지 않음).

- ×1.5가 아니라 ×3인 이유: 실측 검증 결과 ×1.5 기준 경계구간(3배는 안 넘지만 1.5배는 넘는 구간)의 77%가
  2~4인 가족여행이고 1인당 지출은 24만원 수준(전체 평균의 2.7배)으로 정상 범위. COST가 "1인당"이 아니라
  "그룹 총액"이라 인원이 많다는 이유만으로 정상 가족여행이 고액 이상치로 오분류되는 것을 방지하기 위해 ×3 채택.

In [5]:
def find_outlier(col_or_series, name):
    s_all = col_or_series if isinstance(col_or_series, pd.Series) else df[col_or_series]
    flag = pd.Series(False, index=df.index)
    print(f'[{name}]')
    for y in sorted(df['연도'].unique()):
        m = (df['연도'] == y) & s_all.notna() & (s_all.astype('float') > 0)
        s = s_all[m].astype('float')
        q1, q3 = s.quantile([.25, .75])
        upper = q3 + 3 * (q3 - q1)
        hit = m & (s_all.astype('float') > upper)
        flag |= hit
        print(f'  {y}  상한={upper:>12,.0f}  검출={hit.sum():>5,}건')
    return flag

f_cost  = find_outlier('D_TRA1_COST', 'D_TRA1_COST (여행 총경비)')
f_one   = find_outlier('D_TRA1_ONE_COST', 'D_TRA1_ONE_COST (1인 지출)')
f_spend = find_outlier(df[CST].sum(axis=1), '지역 사전집계 지출 합계')

[D_TRA1_COST (여행 총경비)]
  2023  상한=   1,063,000  검출=1,064건
  2024  상한=     964,000  검출=1,232건
  2025  상한=   1,060,000  검출=  977건
[D_TRA1_ONE_COST (1인 지출)]
  2023  상한=     450,000  검출=1,045건
  2024  상한=     450,000  검출=  982건
  2025  상한=     482,000  검출=  972건
[지역 사전집계 지출 합계]
  2023  상한=     450,000  검출=1,065건
  2024  상한=     468,834  검출=  957건
  2025  상한=     549,997  검출=  640건


In [6]:
# 최댓값이 오류인지 실재하는 여행인지 직접 확인
top = df.loc[df['D_TRA1_COST'].astype('float').idxmax()]
print('D_TRA1_COST 최댓값 행: 동반인원', top['D_TRA1_NUM'], '명, 1인지출', f"{top['D_TRA1_ONE_COST']:,}", '원')
print('-> 동반인원이 많은 단체여행. 1인 지출로 환산하면 정상 범위이므로 값을 바꾸지 않는다.')

df['FLAG_OUTLIER_TRIP1_COST']     = f_cost
df['FLAG_OUTLIER_TRIP1_ONE_COST'] = f_one
df['FLAG_OUTLIER_SPEND_TOTAL']    = f_spend

for c in ['FLAG_OUTLIER_TRIP1_COST','FLAG_OUTLIER_TRIP1_ONE_COST','FLAG_OUTLIER_SPEND_TOTAL']:
    print(f'{c:32s} {df[c].sum():>6,}건')

step_log.append(('이상치 처리', df.shape[0], df.shape[1]))
print()
print('원본 값 변경 : 0건 (플래그만 추가)')

D_TRA1_COST 최댓값 행: 동반인원 80 명, 1인지출 300,000 원
-> 동반인원이 많은 단체여행. 1인 지출로 환산하면 정상 범위이므로 값을 바꾸지 않는다.
FLAG_OUTLIER_TRIP1_COST           3,273건
FLAG_OUTLIER_TRIP1_ONE_COST       2,999건
FLAG_OUTLIER_SPEND_TOTAL          2,662건

원본 값 변경 : 0건 (플래그만 추가)


---
## 4. 손상 레코드

1. 산술 정합성 — `COST = NUM × ONE_COST`
2. 시점 정합성 — 조사연도/여행연도, 월 범위
3. 회차 순번 불연속
4. **완전 중복행** — `ID`를 뺀 나머지 컬럼으로 검사 (ID를 포함하면 응답자 고유값이라 검사 자체가 항상 0건이 되어 무의미해짐)

세출 데이터와 반대로, 국민여행조사는 한 행 = 응답자 1명이라 "비슷한 사람이 여럿"인 게 정상이다.
삭제 시뮬레이션 결과 모집단 추정이 10% 무너지는 것을 확인해 **삭제하지 않는다** (진경 검증 결과 그대로 채택).

In [7]:
mis = pd.Series(False, index=df.index)
for n in range(1, 7):
    c, u, o = f'D_TRA{n}_COST', f'D_TRA{n}_NUM', f'D_TRA{n}_ONE_COST'
    ok = df[[c, u, o]].notna().all(axis=1)
    gap_val = (df[c].astype('float') - df[u].astype('float') * df[o].astype('float')).abs()
    mis |= ok & (gap_val > 1)
print(f'산술 정합성(COST=NUM*ONE_COST) 불일치 : {mis.sum()}건 (반올림 잔차, 플래그만)')

bad_year = pd.Series(False, index=df.index)
bad_mon  = pd.Series(False, index=df.index)
for n in range(1, 7):
    sy, sm = df[f'D_TRA{n}_SYEAR'], df[f'D_TRA{n}_SMONTH']
    bad_year |= sy.notna() & (sy.astype('float') != df['연도'])
    bad_mon  |= sm.notna() & ~sm.astype('float').between(1, 12)
print(f'SYEAR != 조사연도 : {bad_year.sum()}건, SMONTH 1~12 이탈 : {bad_mon.sum()}건')

gap = pd.Series(False, index=df.index)
for n in range(2, 7):
    gap |= df[f'D_TRA{n}_SYEAR'].notna() & df[f'D_TRA{n-1}_SYEAR'].isna()
print(f'회차 순번 불연속 : {gap.sum()}건 (병합 오류가 아니라 원자료 회차 슬롯 특성, 플래그만)')

산술 정합성(COST=NUM*ONE_COST) 불일치 : 1건 (반올림 잔차, 플래그만)
SYEAR != 조사연도 : 0건, SMONTH 1~12 이탈 : 0건
회차 순번 불연속 : 440건 (병합 오류가 아니라 원자료 회차 슬롯 특성, 플래그만)


In [8]:
# 완전 중복행: ID를 일부러 빼고 나머지 컬럼으로 검사 (같은 사람이 ID만 다르게 중복 등록됐는지 확인하는 목적)
dup = df.drop(columns=['ID']).duplicated(keep=False)
TRA = [c for c in df.columns if c.startswith('D_TRA')]
non_traveler = df[TRA].isna().all(axis=1) & (df[CNT].sum(axis=1) == 0)

print(f'ID 제외 중복행 : {dup.sum():,}건 (비여행자 {(dup & non_traveler).sum():,} / 여행기록 有 {(dup & ~non_traveler).sum():,})')

# 삭제 시뮬레이션 (실제 적용하지 않음) — 모집단 추정이 얼마나 왜곡되는지 확인
kept = df.drop(columns=['ID']).drop_duplicates()
print(f'[시뮬레이션] drop_duplicates() 적용 시 : {len(df):,}행 -> {len(kept):,}행 ({len(df)-len(kept):,}명 삭제)')
for y in sorted(df['연도'].unique()):
    a = df.loc[df['연도'] == y, 'WT_DOM'].sum() / 12
    b = df.loc[kept.index][lambda x: x['연도'] == y]['WT_DOM'].sum() / 12
    print(f'  {y} 모집단 추정: 현재 {a:,.0f}명 -> 삭제 후 {b:,.0f}명 ({(b-a)/a:+.1%})')
print('-> 모집단 추정이 크게 왜곡되므로 삭제하지 않는다.')

df['FLAG_COST_INCONSISTENT'] = mis
df['FLAG_ROUND_GAP']          = gap
df['FLAG_DUP_SUSPECT']        = dup & ~non_traveler

step_log.append(('손상 레코드 처리', df.shape[0], df.shape[1]))
print()
print(f'행 삭제 : 0건 (현재 {df.shape[0]:,}행)')

ID 제외 중복행 : 26,014건 (비여행자 25,952 / 여행기록 有 62)


[시뮬레이션] drop_duplicates() 적용 시 : 156,050행 -> 140,397행 (15,653명 삭제)
  2023 모집단 추정: 현재 45,872,017명 -> 삭제 후 41,568,624명 (-9.4%)


  2024 모집단 추정: 현재 46,265,820명 -> 삭제 후 41,591,581명 (-10.1%)
  2025 모집단 추정: 현재 46,426,098명 -> 삭제 후 41,830,947명 (-9.9%)
-> 모집단 추정이 크게 왜곡되므로 삭제하지 않는다.

행 삭제 : 0건 (현재 156,050행)


---
## 5. 파생변수

### 5-1. 거주지 복원 (진경)
리사이징 과정에서 빠진 거주지 컬럼을 `ID` 앞 2자리로 복원한다. 광역시 방문 왜곡 문제(트랙1 4분면 분석의 전제) 검증에 필수.

In [9]:
df['RESIDENCE_SIDO']       = df['ID'].str[:2].map(SIDO)
df['RESIDENCE_SIGUNGU_CD'] = df['ID'].str[:5]

# 검증: 관광여행 1건이면서 1개 지역만 방문한 사람의 SPOT 앞 2자리와 거주지가 일치해야 함
tour = df[CASE].isin(TOUR_CASE)
one_trip = (tour.sum(axis=1) == 1) & (df[CNT].sum(axis=1) == 1)
sub = df[one_trip]
spot_mat = sub[[f'D_TRA{n}_1_SPOT' for n in range(1, 7)]].astype('float').values
idx = tour[one_trip].values.argmax(axis=1)
spot = spot_mat[np.arange(len(sub)), idx]
region = [REGIONS[i] for i in sub[CNT].values.argmax(axis=1)]
chk = pd.DataFrame({'SPOT앞2': pd.Series(spot).astype(int).astype(str).str[:2].map(SIDO).values,
                    '사전집계지역': region})
match_rate = (chk['SPOT앞2'] == chk['사전집계지역']).mean()
print(f'RESIDENCE_SIDO 매핑률 : {df["RESIDENCE_SIDO"].notna().mean():.4f}')
print(f'SPOT 교차검증 표본 {len(chk):,}명, 일치율 {match_rate:.4f}')
assert match_rate > 0.999, 'SPOT 교차검증 일치율이 낮음 — 거주지 복원 로직 재확인 필요'

RESIDENCE_SIDO 매핑률 : 1.0000
SPOT 교차검증 표본 69,042명, 일치율 1.0000


### 5-2. 공식 정의의 관광여행 횟수 (진경)

`CASE`가 1·2·4(관광/휴양, 친지방문+관광, 출장+관광)일 때만 관광여행으로 센다.

In [10]:
df['TOUR_TRIP_CNT'] = df[CASE].isin(TOUR_CASE).sum(axis=1).astype('int64')
print('TOUR_TRIP_CNT 분포 :', df['TOUR_TRIP_CNT'].value_counts().sort_index().to_dict())

TOUR_TRIP_CNT 분포 : {0: 83916, 1: 70277, 2: 1743, 3: 86, 4: 24, 5: 2, 6: 2}


### 5-3. SPOT → 시/군/구 디코딩 (신규 작업)

두 원본 노트북 모두 시/도 단위까지만 만들었다. 트랙2 방법론 2-2 ②가 요구하는 시/군/구 단위 방문지를
코드북 "시도 및 시군구 코드" 표(229개 시군구, PDF에서 파싱)를 join해서 만든다.
`D_TRA1_1_SPOT`은 5자리 코드(앞 2자리=시도, 뒤 3자리=시군구)다.

In [11]:
with open(SIGUNGU_MAP_PATH, encoding='utf-8') as f:
    _map = json.load(f)
SIDO_FULLNAME = _map['sido_names']                 # {'11': '서울특별시', ...}
SIGUNGU_MAP = _map['sigungu_map']                  # {'11230': '강남구', ...}
print(f'시군구 코드 매핑 {len(SIGUNGU_MAP)}건 로드 (코드북 파싱 결과, 방법론 문서의 n=229와 일치)')

spot_str = df['D_TRA1_1_SPOT'].astype('float')
spot_5digit = spot_str.dropna().astype(int).astype(str).str.zfill(5).reindex(df.index)

df['TRIP1_DEST_SIGUNGU_CD'] = spot_5digit
sido_part = spot_5digit.str[:2]
sgg_part_key = spot_5digit  # '11230' 형태로 이미 sigungu_map 키와 동일

df['TRIP1_DEST_SIDO'] = sido_part.map(SIDO)
df['TRIP1_DEST_SIGUNGU_NM'] = sgg_part_key.map(SIGUNGU_MAP)

matched = df['TRIP1_DEST_SIGUNGU_CD'].notna()
match_ok = df.loc[matched, 'TRIP1_DEST_SIGUNGU_NM'].notna().mean()
print(f'시/군/구명 매핑 성공률(방문기록 있는 사람 중) : {match_ok:.4f}')
print(df[['D_TRA1_1_SPOT','TRIP1_DEST_SIDO','TRIP1_DEST_SIGUNGU_NM']].dropna().head(5).to_string(index=False))
print()
print('[참고] 매핑 안 된 SPOT 코드 (구 지역 개편 등으로 코드북에 없는 코드일 가능성, 원본 값은 보존됨):')
unmatched = df.loc[matched & df['TRIP1_DEST_SIGUNGU_NM'].isna(), 'D_TRA1_1_SPOT'].unique()
print(f'  {len(unmatched)}개 코드:', sorted(unmatched)[:10], '...' if len(unmatched) > 10 else '')

시군구 코드 매핑 229건 로드 (코드북 파싱 결과, 방법론 문서의 n=229와 일치)
시/군/구명 매핑 성공률(방문기록 있는 사람 중) : 1.0000
 D_TRA1_1_SPOT TRIP1_DEST_SIDO TRIP1_DEST_SIGUNGU_NM
         37330              경북                   청송군
         32010              강원                   춘천시
         31220              경기                   안성시
         32410              강원                   양양군
         34060              충남                   논산시

[참고] 매핑 안 된 SPOT 코드 (구 지역 개편 등으로 코드북에 없는 코드일 가능성, 원본 값은 보존됨):
  0개 코드: [] 


### 5-4. 방문 시/도, 인구통계 라벨, 지역×월 교차변수 (승희)

In [12]:
# 방문 시/도 코드·명 (1차 여행 첫 방문지 기준) — 5-3의 TRIP1_DEST_SIDO 와 동일 개념이므로 그대로 재사용
df['VISIT_SIDO_CD'] = sido_part.astype('Int64')
df['VISIT_SIDO_NM'] = df['TRIP1_DEST_SIDO']

# 인구통계 라벨링 (코드북 원문 그대로)
sex_map = {1: '남자', 2: '여자'}
age_map = {1: '15~19세', 2: '20대', 3: '30대', 4: '40대', 5: '50대', 6: '60대', 7: '70세 이상'}
inc_map = {1: '100만원 미만', 2: '100~200만원 미만', 3: '200~300만원 미만',
           4: '300~400만원 미만', 5: '400~500만원 미만', 6: '500~600만원 미만', 7: '600만원 이상'}
df['SEX_LABEL'] = df['BSEX'].map(sex_map)
df['AGE_LABEL'] = df['BAGE'].map(age_map)
df['INCOME_LABEL'] = df['BINC1'].map(inc_map)

# 지역x월 교차변수 (트랙2 챗봇용 groupby 키)
df['TRIP1_SIDO_MONTH'] = df['VISIT_SIDO_NM'] + '_' + df['D_TRA1_SMONTH'].astype('string') + '월'

# 거주지 지역유형 (신규 추가) — 트랙1 4분면 층화 분석용
df['RESIDENCE_REGION_TYPE'] = df['RESIDENCE_SIDO'].apply(
    lambda x: ('광역시' if x in METRO else '도') if pd.notna(x) else pd.NA)

print(df['SEX_LABEL'].value_counts().to_string())
print()
print(df['RESIDENCE_REGION_TYPE'].value_counts().to_string())
print()
print('고유 지역x월 조합 수:', df['TRIP1_SIDO_MONTH'].nunique(), '(17개 지역 x 12개월 =', 17*12, '이하)')

step_log.append(('파생 변수 생성', df.shape[0], df.shape[1]))

SEX_LABEL
여자    78862
남자    77188

RESIDENCE_REGION_TYPE
도      95273
광역시    60777

고유 지역x월 조합 수: 204 (17개 지역 x 12개월 = 204 이하)


---
## 6. 공식 통계 대조

전처리 결과가 맞는지 확인하는 가장 확실한 방법. 문체부 공식 발표치와 맞춰본다.
`WT_DOM`은 월 단위 가중치이므로 모집단은 `Σ WT_DOM ÷ 12`, 총량은 `Σ(WT_DOM × 값)`이다.

In [13]:
print(f'{"연도":>5}{"모집단(ΣWT/12)":>18}{"관광여행 횟수":>16}{"지출":>16}{"1인평균 횟수":>14}{"1인평균 지출":>14}')
official = {
    2023: dict(trips=None, spend=None, per_trip=5.70, per_spend=771),
    2024: dict(trips=None, spend=None, per_trip=5.56, per_spend=741),
    2025: dict(trips=261299_000, spend=36353_000_000_000, per_trip=5.63, per_spend=783),
}
for y in sorted(df['연도'].unique()):
    g = df[df['연도'] == y]
    w = g['WT_DOM'].values
    pop = w.sum() / 12
    trips = (g['TOUR_TRIP_CNT'].values * w).sum()
    spend = (g[CST].sum(axis=1).values * w).sum()
    print(f'{y:>5}{pop:>17,.0f}명{trips/1e3:>15,.0f}천회{spend/1e9:>13,.0f}십억원'
          f'{trips/pop:>13.2f}회{spend/pop/1000:>13,.0f}천원')

print()
print('공식 수치(2025년 국민여행조사 보고서-분석편): 총량 261,299천회 / 36,353십억원')
print('1인평균 횟수(23/24/25): 5.70 / 5.56 / 5.63회, 1인평균 지출: 771 / 741 / 783천원')
print('2025년 장래인구추계(15세 이상): 46,426,098명')

g2025 = df[df['연도'] == 2025]
pop2025 = g2025['WT_DOM'].sum() / 12
trips2025 = (g2025['TOUR_TRIP_CNT'].values * g2025['WT_DOM'].values).sum()
assert abs(pop2025 - 46_426_098) < 1000, '2025 모집단 추정치가 공식 인구추계와 어긋남'
assert abs(trips2025 / 1000 - 261299) / 261299 < 0.001, '2025 관광여행 총량이 공식치와 어긋남 — 회귀테스트 실패'
print()
print('회귀테스트 통과: 2025년 모집단·관광여행 총량이 공식 수치와 일치')

   연도       모집단(ΣWT/12)         관광여행 횟수              지출       1인평균 횟수       1인평균 지출
 2023       45,872,017명        261,651천회       35,348십억원         5.70회          771천원


 2024       46,265,820명        257,311천회       34,280십억원         5.56회          741천원


 2025       46,426,098명        261,299천회       36,353십억원         5.63회          783천원

공식 수치(2025년 국민여행조사 보고서-분석편): 총량 261,299천회 / 36,353십억원
1인평균 횟수(23/24/25): 5.70 / 5.56 / 5.63회, 1인평균 지출: 771 / 741 / 783천원
2025년 장래인구추계(15세 이상): 46,426,098명

회귀테스트 통과: 2025년 모집단·관광여행 총량이 공식 수치와 일치


---
## 7. 컬럼명 영문화 (승희 방식 채택)

지역별 사전집계 34개 컬럼과 회차 반복필드 42개는 승희의 규칙(`trip_cnt_{region}`, `trip{n}_{field}`)을 그대로 사용하고,
이번에 새로 추가한 파생변수도 같은 snake_case 규칙으로 통일한다.

In [14]:
region_en = {
    '서울':'seoul','부산':'busan','대구':'daegu','인천':'incheon','광주':'gwangju',
    '대전':'daejeon','울산':'ulsan','세종':'sejong','경기':'gyeonggi','강원':'gangwon',
    '충북':'chungbuk','충남':'chungnam','전북':'jeonbuk','전남':'jeonnam',
    '경북':'gyeongbuk','경남':'gyeongnam','제주':'jeju'
}

col_rename = {
    'ID':'id', 'WT_DOM':'wt_dom', 'BSEX':'bsex', 'BAGE':'bage', 'BINC1':'binc1', '연도':'year',
}

round_field_map = {
    'SYEAR':'start_year', 'SMONTH':'start_month', '1_SPOT':'spot_cd',
    'CASE':'case', 'COST':'cost', 'NUM':'num', 'ONE_COST':'cost_per_person'
}
for n in range(1, 7):
    for kr_suffix, en_suffix in round_field_map.items():
        col_rename[f'D_TRA{n}_{kr_suffix}'] = f'trip{n}_{en_suffix}'

for kr, en in region_en.items():
    col_rename[f'국내_A_여행횟수_관광전체_{kr}'] = f'trip_cnt_{en}'
    col_rename[f'국내_A_여행지출_관광전체_{kr}'] = f'trip_exp_{en}'

col_rename.update({
    # 진경 파생 (뼈대)
    'RESIDENCE_SIDO': 'residence_sido',
    'RESIDENCE_SIGUNGU_CD': 'residence_sigungu_cd',
    'TOUR_TRIP_CNT': 'tourism_trip_cnt',
    'FLAG_OUTLIER_TRIP1_COST': 'flag_outlier_trip1_cost',
    'FLAG_OUTLIER_TRIP1_ONE_COST': 'flag_outlier_trip1_one_cost',
    'FLAG_OUTLIER_SPEND_TOTAL': 'flag_outlier_spend_total',
    'FLAG_COST_INCONSISTENT': 'flag_cost_inconsistent',
    'FLAG_ROUND_GAP': 'flag_round_gap',
    'FLAG_DUP_SUSPECT': 'flag_dup_suspect',
    # 신규: SPOT -> 시군구 디코딩
    'TRIP1_DEST_SIGUNGU_CD': 'trip1_dest_sigungu_cd',
    'TRIP1_DEST_SIDO': 'trip1_dest_sido',
    'TRIP1_DEST_SIGUNGU_NM': 'trip1_dest_sigungu_nm',
    # 승희 파생 (표면)
    'VISIT_SIDO_CD': 'visit_sido_cd',
    'VISIT_SIDO_NM': 'visit_sido_nm',
    'SEX_LABEL': 'sex_label',
    'AGE_LABEL': 'age_label',
    'INCOME_LABEL': 'income_label',
    'TRIP1_SIDO_MONTH': 'trip1_sido_month',
    # 신규: 거주지 지역유형(광역시/도) 층화용
    'RESIDENCE_REGION_TYPE': 'residence_region_type',
})

missing = set(df.columns) - set(col_rename.keys())
assert not missing, f'매핑 안 된 컬럼 발견: {missing}'
extra = set(col_rename.keys()) - set(df.columns)
assert not extra, f'df에 없는데 매핑에 있는 항목: {extra}'

en_names = list(col_rename.values())
dups = {x for x in en_names if en_names.count(x) > 1}
assert not dups, f'영문명 중복: {dups}'

df = df.rename(columns=col_rename)
print(df.shape)
print('영문 컬럼명 검증 통과, 중복 없음')

(156050, 101)
영문 컬럼명 검증 통과, 중복 없음


---
## 8. 저장 및 검증

In [15]:
df = df.reset_index(drop=True)
df.to_csv(DST, index=False, encoding='utf-8-sig')

step_log.append(('최종(processed_combined.csv)', df.shape[0], df.shape[1]))

summary = pd.DataFrame(step_log, columns=['단계', '행 수', '컬럼 수'])
summary['행 증감'] = summary['행 수'].diff().fillna(0).astype(int)
summary['컬럼 증감'] = summary['컬럼 수'].diff().fillna(0).astype(int)
print(summary.to_string(index=False))
print()
print(f'{DST} 저장 완료: {df.shape[0]:,}행 x {df.shape[1]}열')

                        단계    행 수  컬럼 수  행 증감  컬럼 증감
                        원본 156050    82     0      0
                    결측치 처리 156050    82     0      0
                     타입 변환 156050    82     0      0
                    이상치 처리 156050    85     0      3
                 손상 레코드 처리 156050    88     0      3
                  파생 변수 생성 156050   101     0     13
최종(processed_combined.csv) 156050   101     0      0

processed_combined.csv 저장 완료: 156,050행 x 101열


### 검증 — 원본 값이 하나도 안 바뀌었는지 재확인

원본 82개 컬럼(dtype 변환 대상 제외)의 수치가 저장 후에도 동일한지, 행 순서가 그대로인지 확인한다.

In [16]:
orig = pd.read_csv(SRC, low_memory=False)
chk  = pd.read_csv(DST, low_memory=False)

print('행 수 동일          :', len(chk) == len(orig))
print('id 순서 동일        :', (chk['id'] == orig['ID']).all())

# 원본 컬럼명 -> 영문명 역매핑으로 대조 (dtype 변환된 회차/지출 컬럼은 반올림 오차 허용 범위 내에서 비교)
inv_rename = {v: k for k, v in col_rename.items()}
num_checked = 0
changed = []
for en_col in chk.columns:
    kr_col = inv_rename.get(en_col)
    if kr_col in orig.columns and pd.api.types.is_numeric_dtype(orig[kr_col]):
        num_checked += 1
        a = orig[kr_col].fillna(-9e9).astype(float)
        b = pd.to_numeric(chk[en_col], errors='coerce').fillna(-9e9).astype(float)
        if not np.allclose(a, b, atol=1):
            changed.append(en_col)

print(f'원본과 대조한 숫자 컬럼 : {num_checked}개')
print('값이 원본과 달라진 컬럼 :', changed if changed else '없음')

행 수 동일          : True
id 순서 동일        : True
원본과 대조한 숫자 컬럼 : 81개
값이 원본과 달라진 컬럼 : 없음
